In [ ]:
# ===== S10: exp-029 slice-bag fold-0 SOLO (user-approved) =====
import os
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn

t0 = time.time()
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


def log(m):
    print(f"[S10 {time.time() - t0:7.1f}s] {m}", flush=True)


def uptime():
    try:
        return float(Path("/proc/uptime").read_text().split()[0])
    except Exception:
        return 0.0


for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
          Path("/kaggle/input/rsna-knee-abnormality-detection")]:
    if (c / "test_series").is_dir():
        ROOT = c
        break
sample = pd.read_csv(ROOT / "sample_submission.csv",
                     dtype={"StudyInstanceUID": str})
sample.to_csv("/kaggle/working/submission.csv", index=False)   # fallback
log(f"fallback submission written ({len(sample)} rows)")


GROUPS_BY = "run"


_PSA = None


def _psa_call(study):
    return _PSA(study)


def _psa_warm(i):
    return i


_BLOBS = {}
_IOP = None


def _io_pool():
    global _IOP
    if _IOP is None:
        from concurrent.futures import ThreadPoolExecutor as _TP
        _IOP = _TP(max_workers=16)
    return _IOP


def _read_bytes(p):
    with open(p, "rb") as f:
        return f.read()


def run_s10():
    import os                          # fast kernel: bind os before the fork (a later import makes it local)
    IMG512, CROP_MM, LAT_MIN_OFFSET_MM = 512, 160.0, 5.0
    NEED224, NEED336, CAP = False, True, 96   # fast kernel: fixed at build
    # exp-084 trained on allslice224: central 480 of the 512px/160mm arrays
    # = 150 mm, resized to 224 px (build.py:29 IMG, CROP = 224, 480).
    CROP480, IMG224 = 480, 224
    O480 = (IMG512 - CROP480) // 2
    BAG, N_BAGS, BAG_SEED = 32, 4, 12345
    TIME_CAP_S = 8.0 * 3600

    inp = Path("/kaggle/input")
    log(f"input dirs: {sorted(p.name for p in inp.iterdir())}")

    def _kglob(name, recursive=True):
        roots = [q for q in list(inp.glob("datasets/*/*")) + list(inp.glob("*")) if q.is_dir()
                 and q.name not in ("datasets", "competitions", "rsna-knee-abnormality-detection")]
        return sorted({str(h) for r in roots for h in r.rglob(name)})
    _dsdirs = [q for q in list(inp.glob("datasets/*/*")) + list(inp.glob("models/*/*")) + list(inp.glob("*"))
               if q.is_dir() and "rsna-camp" in q.name]
    CKPTS = sorted({p.resolve() for d in _dsdirs for p in d.rglob("*.pt")})
    assert CKPTS, "no campaign checkpoints mounted"
    log(f"{len(CKPTS)} checkpoints: {[p.name for p in CKPTS]}")

    ts = pd.read_csv(ROOT / "test_series.csv")
    PLANE_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}
    S_PLANE = dict(zip(ts["SeriesInstanceUID"].astype(str),
                       ts["Anatomical_Plane"].astype(str)))
    S_FLUID = dict(zip(ts["SeriesInstanceUID"].astype(str),
                       ts["Fluid_Sensitive"].astype(int)))
    S_FATSAT = dict(zip(ts["SeriesInstanceUID"].astype(str),
                        ts["Fat_Suppression"].astype(int)))

    # ---- per-study preprocessing: EXACT port of prep_full_template.py
    SORT_TAGS = ["ImagePositionPatient", "InstanceNumber", "Laterality",
                 "ImageLaterality", "PixelSpacing"]

    def read_headers(sdir, files):
        out = []
        for name in files:
            h = {"file": name}
            try:
                _p = os.path.join(sdir, name)
                _b = _BLOBS.pop(_p, None)
                d = pydicom.dcmread(__import__('io').BytesIO(_b) if _b is not None else _p, force=True)
                h["ds"] = d                          # fast kernel: pixel pass reuses this read
            except Exception:
                out.append(h)
                continue
            try:
                ipp = getattr(d, "ImagePositionPatient", None)
                if ipp is not None and len(ipp) >= 3:
                    h["ipp"] = np.asarray([float(v) for v in ipp[:3]])
            except Exception:
                pass
            try:
                inst = getattr(d, "InstanceNumber", None)
                h["instance"] = None if inst is None else float(inst)
            except Exception:
                h["instance"] = None
            try:
                for t in ("ImageLaterality", "Laterality"):
                    v = getattr(d, t, None)
                    if v and str(v).strip().upper()[:1] in ("L", "R"):
                        h["lat"] = str(v).strip().upper()[:1]
                        break
            except Exception:
                pass
            try:
                ps = getattr(d, "PixelSpacing", None)
                if ps is not None and len(ps) >= 2:
                    h["ps"] = (float(ps[0]), float(ps[1]))
            except Exception:
                pass
            out.append(h)
        return out

    def sort_order(headers):
        pos = [h for h in headers if "ipp" in h]
        if len(pos) >= max(2, int(0.8 * len(headers))):
            xyz = np.stack([h["ipp"] for h in pos])
            axis = int(np.argmax(np.ptp(xyz, axis=0)))
            fb = float(np.median(xyz[:, axis]))
            key = lambda h: (float(h["ipp"][axis]) if "ipp" in h else fb,
                             h["instance"] if h.get("instance") is not None
                             else float("inf"))
        elif sum(h.get("instance") is not None for h in headers) >= 2:
            key = lambda h: (h["instance"] if h.get("instance") is not None
                             else float("inf"))
        else:
            key = lambda h: h["file"]
        return sorted(headers, key=key)

    def decode_series(sdir, headers):
        planes, shapes = [], []
        px_r = px_c = None
        for h in headers:
            try:
                ds = h.pop("ds", None)
                if ds is None:
                    ds = pydicom.dcmread(os.path.join(sdir, h["file"]), force=True)
                a = ds.pixel_array
                if a.ndim != 2:
                    planes.append(None)
                    shapes.append(None)
                    continue
                a = a.astype(np.float32)
                slope = float(getattr(ds, "RescaleSlope", 1) or 1)
                inter = float(getattr(ds, "RescaleIntercept", 0) or 0)
                if slope != 1.0:
                    a *= np.float32(slope)
                if inter != 0.0:
                    a += np.float32(inter)
                if str(getattr(ds, "PhotometricInterpretation",
                               "MONOCHROME2")) == "MONOCHROME1":
                    a = -a
                planes.append(a)
                shapes.append(a.shape)
            except Exception:
                planes.append(None)
                shapes.append(None)
        good_shapes = [s for s in shapes if s is not None]
        if not good_shapes:
            return None, None, None, None
        shp = Counter(good_shapes).most_common(1)[0][0]
        fixed, good = [], []
        for p in planes:
            if p is None or p.shape != shp:
                fixed.append(np.zeros(shp, np.float32))
                good.append(False)
            else:
                fixed.append(p)
                good.append(True)
        vol = np.stack(fixed)
        pss = [h["ps"] for h in headers if "ps" in h]
        if pss:
            px_r = float(np.median([p[0] for p in pss]))
            px_c = float(np.median([p[1] for p in pss]))
        return vol, np.array(good, bool), px_r, px_c

    def crop_pad_axis(vol, px, axis):
        if not (px and np.isfinite(px) and px > 0):
            return vol, 0
        want = int(round(CROP_MM / px))
        n = vol.shape[axis]
        if want < 16:
            return vol, 0
        if want < n:
            start = (n - want) // 2
            sl = [slice(None)] * 3
            sl[axis] = slice(start, start + want)
            return vol[tuple(sl)], 0
        if want > n:
            return vol, want - n
        return vol, 0

    def process_series(vol, good, px_r, px_c):
        vol, pad_r = crop_pad_axis(vol, px_r, 1)
        vol, pad_c = crop_pad_axis(vol, px_c, 2)
        gv = vol[good]
        lo, hi = np.percentile(gv.reshape(-1)[::7], [1, 99])   # fast kernel: strided
        vol = np.ascontiguousarray(vol, dtype=np.float32)
        np.subtract(vol, np.float32(lo), out=vol)
        np.multiply(vol, np.float32(1.0 / max(hi - lo, 1e-6)), out=vol)
        np.clip(vol, 0.0, 1.0, out=vol)
        if pad_r or pad_c:
            vol = np.pad(vol, ((0, 0),
                               (pad_r // 2, pad_r - pad_r // 2),
                               (pad_c // 2, pad_c - pad_c // 2)))
        interp = (cv2.INTER_AREA if min(vol.shape[1:]) > IMG512
                  else cv2.INTER_LINEAR)
        out = np.empty((vol.shape[0], IMG512, IMG512), np.float32)
        for i in range(vol.shape[0]):
            out[i] = cv2.resize(vol[i], (IMG512, IMG512),
                                interpolation=interp)
        np.multiply(out, np.float32(255), out=out)
        np.rint(out, out=out)
        np.clip(out, 0, 255, out=out)
        u8 = out.astype(np.uint8)
        u8[~good] = 0
        return u8

    def study_laterality(series_infos):
        ballots = []
        for si in series_infos:
            lats = {h["lat"] for h in si["headers"] if h.get("lat")}
            if len(lats) == 1:
                ballots.append(next(iter(lats)))
            elif len(lats) > 1:
                ballots.append("conflict")
        votes = [b for b in ballots if b in ("L", "R")]
        tag_side = None
        if votes:
            cl, cr = votes.count("L"), votes.count("R")
            if cl != cr:
                tag_side = "L" if cl > cr else "R"
        xs = []
        for si in series_infos:
            if si["plane"] == "Sagittal":
                sx = [float(h["ipp"][0]) for h in si["headers"] if "ipp" in h]
                if sx:
                    xs.append(float(np.median(sx)))
        ippx_side = None
        if xs:
            x = float(np.median(xs))
            if abs(x) >= LAT_MIN_OFFSET_MM:
                ippx_side = "R" if x < 0 else "L"
        if tag_side:
            return tag_side
        if ippx_side:
            return ippx_side
        if votes:
            return sorted(set(votes))[0]
        return None

    def normalise_laterality(vol, plane, lat):
        if lat != "R":
            return vol
        if plane in ("Coronal", "Axial"):
            return vol[:, :, ::-1].copy()
        if plane == "Sagittal":
            return vol[::-1].copy()
        return vol



    def process_study_allslice(study):
        """-> (study, imgs [N,224,224] u8 incl. zero slices, meta [N,4] f32,
        jn [N,2] (idx-in-series, n-in-series), nonzero [N] bool) or Nones."""
        try:
            base = ROOT / "test_series" / study
            sdirs = sorted(p for p in base.iterdir() if p.is_dir())
            infos = []
            for sd in sdirs:                       # sorted uid order = cache
                files = sorted(e.name for e in os.scandir(sd)
                               if e.name.endswith(".dcm"))
                if files:
                    infos.append({"uid": sd.name, "dir": str(sd),
                                  "plane": S_PLANE.get(sd.name, "Unknown"),
                                  "files": files})
            _paths = [os.path.join(si["dir"], f) for si in infos for f in si["files"]]
            _BLOBS.clear()
            _BLOBS.update(zip(_paths, _io_pool().map(_read_bytes, _paths)))
            for si in infos:
                si["headers"] = sort_order(read_headers(si["dir"],
                                                        si["files"]))
            lat = study_laterality(infos)
            imgs, imgs336, meta, jn = [], [], [], []
            _sid = -1                       # delivery P1: series index (hier members, meta col 4)
            for si in infos:
                vol, good, px_r, px_c = decode_series(si["dir"],
                                                      si["headers"])
                if vol is None:
                    continue
                u8 = process_series(vol, good, px_r, px_c)
                _sid += 1
                del vol
                u8 = normalise_laterality(u8, si["plane"], lat)
                n = u8.shape[0]
                pl = PLANE_ID.get(si["plane"], 3)
                fl = max(0, S_FLUID.get(si["uid"], 0))
                fs = max(0, S_FATSAT.get(si["uid"], 0))
                for j in range(n):
                    if NEED224:
                        imgs.append(cv2.resize(u8[j, O480:O480 + CROP480, O480:O480 + CROP480],
                                               (IMG224, IMG224), interpolation=cv2.INTER_AREA))
                    if NEED336:                  # 130 mm = central 416 of 512/160 mm
                        imgs336.append(cv2.resize(u8[j, 48:464, 48:464], (336, 336),
                                                  interpolation=cv2.INTER_AREA))
                    meta.append((pl, fl, fs, j / max(n - 1, 1), _sid))
                    jn.append((j, n))
                del u8
            ref = imgs if NEED224 else imgs336
            if not ref:
                return study, None, None, None, None
            views = {}
            if NEED224:
                views[224] = np.stack(imgs)
            if NEED336:
                views[336] = np.stack(imgs336)
            return (study, views, np.asarray(meta, np.float32), np.asarray(jn, np.int32),
                    np.array([im.any() for im in ref], bool))
        except Exception as exc:
            log(f"study fail {study}: {type(exc).__name__} {str(exc)[:80]}")
            return study, None, None, None, None

    global _PSA
    _PSA = process_study_allslice
    import multiprocessing as _mp
    from concurrent.futures import ProcessPoolExecutor as _PPE
    NPROC = max(2, os.cpu_count() or 4)
    _ppool = _PPE(max_workers=NPROC, mp_context=_mp.get_context("fork"))
    list(_ppool.map(_psa_warm, range(4 * NPROC)))       # spawn every worker now, before CUDA exists
    log(f"decode process pool: {NPROC} workers")

    # ---- model: the REAL trainer code, imported (not re-typed).
    # A hand-port of _TimmHybridWrap could drift numerically in a way that is
    # indistinguishable from a family result, so import it instead.
    import glob
    import os
    import subprocess
    import sys as _sys
    import zipfile

    os.environ["TIMM_NO_PRETRAIN"] = "1"   # build the arch; weights are OURS
    whl = _kglob("timm-1.0.11-py3-none-any.whl",
                    recursive=True)
    if whl:
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q",
                        "--no-index", "--no-deps", whl[0]], check=False)
    code = Path("/kaggle/working/code")
    code.mkdir(exist_ok=True, parents=True)
    pz = _kglob("payload.zip", recursive=True)
    if pz:
        with zipfile.ZipFile(pz[0]) as z:
            z.extractall(code)
    else:                                   # Kaggle auto-extracted the zip
        hit = _kglob("train_slicebag.py", recursive=True)
        assert hit, "payload not mounted"
        base = Path(hit[0]).parent
        for f in base.rglob("*"):
            if f.is_file():
                d = code / f.relative_to(base)
                d.parent.mkdir(parents=True, exist_ok=True)
                d.write_bytes(f.read_bytes())
    _sys.path.insert(0, str(code))
    import timm as _timm
    import types as _types
    _cm0 = _timm.create_model

    def _cm(*a, **k):                 # campaign v2 adapter: payload wrap needs .stages
        m = _cm0(*a, **k)
        if not hasattr(m, "stages"):
            if hasattr(m, "layer1"):
                object.__setattr__(m, "stages", [_types.SimpleNamespace(blocks=getattr(m, n))
                                                 for n in ("layer1", "layer2", "layer3", "layer4")])
            elif hasattr(m, "blocks"):
                object.__setattr__(m, "stages", [_types.SimpleNamespace(blocks=st) for st in m.blocks])
        return m
    _timm.create_model = _cm
    # DINOv2 members: no internet on Kaggle -> build from the pinned small config; the
    # checkpoint's strict state_dict load then supplies every weight.
    import transformers as _tf
    _afp0 = _tf.AutoModel.from_pretrained

    def _afp(name, *a, **k):
        if "dinov2-small" in str(name):
            cfg = _tf.Dinov2Config(hidden_size=384, num_hidden_layers=12, num_attention_heads=6,
                                   mlp_ratio=4, patch_size=14, image_size=518, layerscale_value=1.0,
                                   layer_norm_eps=1e-6, qkv_bias=True, use_swiglu_ffn=False,
                                   hidden_act="gelu", drop_path_rate=0.0)
            return _tf.Dinov2Model(cfg)
        return _afp0(name, *a, **k)
    _tf.AutoModel.from_pretrained = _afp
    log(f"timm {_timm.__version__} | code at {code}")
    from eval_study336 import build_model          # -> train_slicebag model
    N_TARGET = len(TARGETS)

    amp = (torch.bfloat16 if torch.cuda.is_available()
           and torch.cuda.is_bf16_supported() else torch.float16)
    NGPU = max(1, torch.cuda.device_count())
    DEVS = [torch.device(f"cuda:{i}") for i in range(NGPU)] if torch.cuda.is_available() \
        else [torch.device("cpu")]
    dev = DEVS[0]
    CHUNK = 32

    _V9 = _kglob("train_slicebag_campaign_v9.py")
    _OFFLINE_SUBS = (("assert pre and pretrained_file, 'campaign requires pinned generic pretrained weights'",
                      "pass"),
                     ("self.m = timm.create_model(name, pretrained=True, num_classes=0,",
                      "self.m = timm.create_model(name, pretrained=pre, num_classes=0,"))
    if _V9:                           # stack4 members: trainer v9 (FiLM / tdec / inst_head / ms_xattn / hier)
        _s9 = Path(_V9[0]).read_text(encoding="utf-8")   # offline: build the arch, weights are OURS
        for _a, _b in _OFFLINE_SUBS:
            assert _s9.count(_a) == 1, _a
            _s9 = _s9.replace(_a, _b)
        (code / "train_slicebag_campaign_v9.py").write_text(_s9, encoding="utf-8")
        import train_slicebag_campaign_v9 as _t9
    _V1G = _kglob("train_slicebag_campaign_v1_gm.py")
    if _V1G:                          # group-mix members: campaign v1 trainer + target-group mixer patch
        _s1 = Path(_V1G[0]).read_text(encoding="utf-8")
        for _a, _b in _OFFLINE_SUBS:
            assert _s1.count(_a) == 1, _a
            _s1 = _s1.replace(_a, _b)
        (code / "train_slicebag_campaign_v1_gm.py").write_text(_s1, encoding="utf-8")
        import train_slicebag_campaign_v1_gm as _t1g
    V9_KEYS = ("ms_xattn", "tdec", "film", "inst_head", "hier")

    def build_any(ck, d):
        if any(str(k_).startswith("gmixers.") for k_ in ck["state_dict"]):
            assert _V1G, "group-mix checkpoint but train_slicebag_campaign_v1_gm.py not mounted"
            assert not any(ck.get(k, 0) for k in V9_KEYS), "group-mix + v9 keys: no builder for this combination"
            _t1g.SliceBagModel.group_mix_default = 1
            m = _t1g.SliceBagModel(ck["backbone"], unfreeze_last=0, d_model=int(ck.get("d_model", 384)),
                                   n_mix=int(ck.get("n_mix", 2)), pool=ck.get("pool", "cls_mean"),
                                   quad_pool=int(ck.get("quad_pool", 0)),
                                   use_xattn=ck.get("decoder", "mil") == "xattn")
            assert m.group_mix == 1
            m.load_state_dict(ck["state_dict"], strict=True)
            m.zoom_px = int(ck.get("zoom_px", 0))
            log(f"build_any v1_gm: {ck['backbone']} group_mix=1")
            return m.to(d).eval()
        if not any(ck.get(k, 0) for k in V9_KEYS):
            return build_model(ck, d)
        assert _V9, "v9 checkpoint but train_slicebag_campaign_v9.py not mounted"
        kw = {k: int(ck[k]) for k in ("ms_xattn", "tdec", "film", "hier") if ck.get(k, 0)}
        if ck.get("inst_head", 0):
            kw["inst_head"] = float(ck["inst_head"])
        m = _t9.SliceBagModel(ck["backbone"], unfreeze_last=0, d_model=int(ck.get("d_model", 384)),
                              n_mix=int(ck.get("n_mix", 2)), pool=ck.get("pool", "cls_mean"),
                              quad_pool=int(ck.get("quad_pool", 0)),
                              use_xattn=ck.get("decoder", "mil") == "xattn", **kw)
        m.load_state_dict(ck["state_dict"], strict=True)
        m.zoom_px = int(ck.get("zoom_px", 0))
        log(f"build_any v9: {ck['backbone']} {kw}")
        return m.to(d).eval()

    def chunked(model):
        enc = model.encode

        def f(x, return_patches=False):
            n, outs, pats, s3s = x.shape[0], [], [], []
            cond = getattr(model, "_film_cond", None)        # v9 FiLM: per-image code, chunk it too
            for i in range(0, n, CHUNK):
                xb = x[i:i + CHUNK]
                k = xb.shape[0]
                if k < CHUNK:
                    xb = torch.cat([xb, xb.new_zeros((CHUNK - k,) + tuple(xb.shape[1:]))])
                if cond is not None:
                    cb = cond[i:i + CHUNK]
                    model._film_cond = cb if k == CHUNK else torch.cat(
                        [cb, cb.new_zeros((CHUNK - k,) + tuple(cb.shape[1:]))])
                if return_patches:                # xattn decoder members need patch tokens
                    o, p = enc(xb, return_patches=True)
                    outs.append(o[:k])
                    pats.append(p[:k])
                else:
                    outs.append(enc(xb)[:k])
                if getattr(model, "ms_xattn", 0):  # v9 ms_xattn: stage-3 map is hook-captured per call
                    s3s.append(model._v9_s3[:k])
            model._film_cond = cond
            if s3s:
                model._v9_s3 = torch.cat(s3s)
            return (torch.cat(outs), torch.cat(pats)) if return_patches else torch.cat(outs)
        model.encode = f
        return model

    MODELS = []                       # (group, device, model, img); fast kernel: one replica per GPU
    GROUP_ALIAS = {'exp225_acqaug_s2000': 'aira_6df7bcc423_s2000'}                  # delivery P6: merge split families into one group
    import re as _re
    for i, cp in enumerate(CKPTS):
        ck = torch.load(cp, map_location="cpu", weights_only=False)
        grp = (ck.get("backbone", "?") if GROUPS_BY == "backbone" else
               _re.sub(r"_f\d+$", "", cp.stem) if GROUPS_BY == "run" else cp.stem)
        grp = GROUP_ALIAS.get(grp, grp)
        img = int(ck.get("img", 224))
        assert img in (224, 336), img
        assert (img == 224 and NEED224) or (img == 336 and NEED336), f"view {img} not built (--views)"
        for d in DEVS:
            m = chunked(build_any(ck, d))
            MODELS.append((grp, d, m, img))
            log(f"model {cp.name}: group {grp} img {img} on {d}")
        del ck
    GROUPS = sorted({g for g, _, _, _ in MODELS})
    MEMBER = {}                       # replica index -> member index (same ckpt on another GPU)
    _seen = {}
    for k, (g_, d_, m_, im_) in enumerate(MODELS):
        MEMBER[k] = k // len(DEVS)
    NEED336 = any(im == 336 for _, _, _, im in MODELS)
    log(f"views: 224{' + 336' if NEED336 else ''}")
    log(f"groups: {GROUPS}")
    THIN_RULES = [('a3_a57ba94073_', 60, 32)]                  # (ckpt file-name prefix, thin_thr, thin_target); the ckpt does not store them
    MEMBER_THIN = {}
    for _i, _cp in enumerate(CKPTS):
        MEMBER_THIN[_i] = next(((_t, _g) for (_p, _t, _g) in THIN_RULES if _cp.name.startswith(_p)), None)
    for (_p, _t, _g) in THIN_RULES:
        assert any(c.name.startswith(_p) for c in CKPTS), f"thin rule {_p} matched no checkpoint"
    assert not any("a57ba94073" in c.name and MEMBER_THIN[_i] is None for _i, c in enumerate(CKPTS)), \
        "an a57ba94073 checkpoint is mounted without its thin-series rule"
    log(f"thin-series members: {sorted(c.name for _i, c in enumerate(CKPTS) if MEMBER_THIN[_i])}")

    def _probe_exec():
        """Per-model precision: fp16 where it runs AND matches fp32 (T4 has no bf16 and no
        fp16 depthwise-conv engine, so ConvNeXt-like members stay fp32)."""
        out = []
        g = torch.Generator().manual_seed(0)
        for gi, d, m, im in MODELS:
            ti = torch.randint(0, 255, (1, 8, 3, im, im), generator=g, dtype=torch.uint8).to(d)
            tm = torch.zeros((1, 8, 5), dtype=torch.float32, device=d)
            tk = torch.ones((1, 8), dtype=torch.float32, device=d)
            with torch.no_grad():
                ref = torch.sigmoid(m(ti, tm, tk).float())
            chosen = None
            for dt in (torch.float16,):
                try:
                    with torch.no_grad(), torch.autocast(d.type, dtype=dt, enabled=d.type == "cuda"):
                        z = torch.sigmoid(m(ti, tm, tk).float())
                    if torch.isfinite(z).all() and float((z - ref).abs().max()) < 5e-3:
                        chosen = dt
                except RuntimeError:
                    torch.cuda.empty_cache()
            out.append(chosen)
            log(f"precision {gi}: {'fp16' if chosen is not None else 'fp32'}")
        return out

    AMP_DT = _probe_exec()

    def thin_gap_rows(n, thr, target):
        """Per-row stride g >= 1: round(n/target) for series with n > thr, else 1 (trainer thin_series)."""
        n = np.asarray(n, np.int64)
        if not thr:
            return np.ones_like(n)
        g = np.where(n > int(thr), np.maximum(1, np.rint(n / float(target))), 1)
        return g.astype(np.int64)

    def thin_select(rows, jn, thr, target):
        """rows: int array of nonzero row ids of ONE study (concatenated series); jn[:, 0]=j, jn[:, 1]=n.
        Returns the rows kept by the trainer's thin_series (centre phase); all rows if none would be kept."""
        rows = np.asarray(rows)
        if not thr or len(rows) == 0:
            return rows
        j = np.asarray(jn)[rows, 0].astype(np.int64)
        n = np.asarray(jn)[rows, 1].astype(np.int64)
        g = thin_gap_rows(n, thr, target)
        keep = (j % g) == (g // 2)
        return rows[keep] if keep.any() else rows

    def tri_idx(rows, jn, thr=0, target=32, tri_gap=1):
        """Row ids (a, b) of the two side channels of each row's 2.5D triplet [arr[a], arr[r], arr[b]].
        thr=0 reproduces the shipped kernel triplet (gap 1); thr>0 is the a57ba94073 thin-series gap."""
        rows = np.asarray(rows)
        j = np.asarray(jn)[rows, 0].astype(np.int64)
        n = np.asarray(jn)[rows, 1].astype(np.int64)
        g = np.full(len(rows), int(tri_gap), np.int64)
        if thr:
            g = np.where(n > int(thr), np.maximum(int(tri_gap), np.rint(n / float(target)).astype(np.int64)), g)
        a = rows - np.minimum(g, j)
        b = rows + np.minimum(g, n - 1 - j)
        return a, b

    def triplet(imgs, r, j, n):
        a = r - 1 if j > 0 else r
        b = r + 1 if j < n - 1 else r
        return np.stack([imgs[a], imgs[r], imgs[b]])

    studies = sorted(p.name for p in (ROOT / "test_series").iterdir()
                     if p.is_dir())
    NS = len(studies)
    log(f"{NS} test studies")
    NMEM = len(CKPTS)
    probs = {k: np.full((NS, N_TARGET), np.nan, np.float64) for k in range(NMEM)}
    from concurrent.futures import ThreadPoolExecutor as _TPE
    dev_pools = [_TPE(max_workers=1) for _ in DEVS]

    def _views_batch(imgs, jn, rows, thin=None):
        a, b = tri_idx(rows, jn, *(thin if thin else (0, 32)))      # thin=None: the shipped +-1 triplet
        return {im: np.stack([arr[a], arr[rows], arr[b]], axis=1)[None] for im, arr in imgs.items()}

    @torch.no_grad()
    def infer_on(di, sj, imgs, meta, jn, nonzero):
        rows0 = np.where(nonzero)[0]
        if len(rows0) == 0:
            return
        d = DEVS[di]
        _inp = {}                         # row rule (None | (thr, target)) -> device tensors
        for k, (_, dk, m, im) in enumerate(MODELS):
            if dk != d:
                continue
            th = MEMBER_THIN[MEMBER[k]]
            if th not in _inp:
                rows = thin_select(rows0, jn, *th) if th else rows0
                if len(rows) > CAP:
                    rows = rows[np.linspace(0, len(rows) - 1, CAP).astype(int)]
                L = len(rows)
                bis = _views_batch(imgs, jn, rows, th)
                tis = {i_: torch.from_numpy(b).pin_memory().to(d, non_blocking=True) for i_, b in bis.items()}
                tm = torch.from_numpy(meta[rows][None].astype(np.float32)).to(d)
                tk = torch.ones((1, L), dtype=torch.float32, device=d)
                _inp[th] = (tis, tm, tk)
            tis, tm, tk = _inp[th]
            dt_k = AMP_DT[k]
            with torch.autocast(d.type, dtype=dt_k or torch.float32,
                                enabled=(dt_k is not None and d.type == "cuda")):
                z = m(tis[im], tm, tk).float()
            probs[MEMBER[k]][sj] = torch.sigmoid(z)[0].cpu().numpy()

    def infer_study(sj, imgs, meta, jn, nonzero):
        di = sj % len(DEVS)
        return dev_pools[di].submit(infer_on, di, sj, imgs, meta, jn, nonzero)

    sidx = {s: i for i, s in enumerate(studies)}
    n_done = 0
    pool = _ppool
    gpu_futs = []
    pending, it = [], iter(studies)
    for _ in range(3 * NPROC):
        s = next(it, None)
        if s:
            pending.append(pool.submit(_psa_call, s))
    while pending:
        fut = pending.pop(0)
        study, imgs, meta, jn, nz = fut.result()
        s = next(it, None)
        if s:
            pending.append(pool.submit(_psa_call, s))
        if imgs is not None:
            gpu_futs.append(infer_study(sidx[study], imgs, meta, jn, nz))
            while sum(not f.done() for f in gpu_futs) > 4 * len(DEVS):   # backpressure on host RAM
                time.sleep(0.005)
        n_done += 1
        if n_done % 100 == 0:
            log(f"{n_done}/{NS} studies decoded ({(time.time() - t0) / max(n_done, 1):.2f} s/study)")
        if uptime() > TIME_CAP_S:
            log("TIME CAP - filling remaining with column medians")
            break
    for f in gpu_futs:
        f.result()
    def _fill(P):
        med = np.nanmedian(P, axis=0)
        for j in range(N_TARGET):
            col = P[:, j]
            col[~np.isfinite(col)] = med[j] if np.isfinite(med[j]) else 0.5
        return P
    # council (gpt-5.6-sol): never let one failed member silently median-fill a study. Members
    # that failed on a study are EXCLUDED from that study's group mean (nan-aware); only studies
    # that no member of a group predicted are median-filled, and every member's gaps are logged.
    per_model_bad = [int((~np.isfinite(probs[k]).all(1)).sum()) for k in range(NMEM)]
    log(f"per-model incomplete studies: {per_model_bad}")
    import warnings as _w
    from scipy.stats import rankdata as _rk
    gsum, n_missing = [], 0
    for g in GROUPS:
        with _w.catch_warnings():
            _w.simplefilter("ignore", category=RuntimeWarning)
            P = np.nanmean(np.stack([probs[MEMBER[k]] for k, (gg, _, _, _) in enumerate(MODELS)
                                     if gg == g and k % len(DEVS) == 0]), axis=0)
        n_missing = max(n_missing, int((~np.isfinite(P).all(1)).sum()))
        P = _fill(P)
        gsum.append(np.column_stack([_rk(P[:, j]) / len(P) for j in range(N_TARGET)]))
    log(f"inference done; {n_missing} studies median-filled")
    probs = np.mean(gsum, axis=0) if len(gsum) > 1 else gsum[0] * 0 + _fill(np.nanmean(
        np.stack([probs[k] for k in range(NMEM)]), axis=0))
    ours = pd.DataFrame(probs, columns=TARGETS)
    ours.insert(0, "StudyInstanceUID", studies)
    out = sample[["StudyInstanceUID"]].merge(ours, on="StudyInstanceUID",
                                             how="left")
    assert len(out) == len(sample) and np.isfinite(
        out[TARGETS].to_numpy()).all()
    out.to_csv("/kaggle/working/submission.csv", index=False)
    log(f"SUBMISSION WRITTEN (ensemble of {len(MODELS)} models, {len(GROUPS)} groups)")


try:
    run_s10()
except Exception as _exc:
    print(f"[S10] FALLBACK kept (sample submission): "
          f"{type(_exc).__name__}: {_exc}", flush=True)
